[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlog7997/rag-playbook/blob/master/learn/02-wrong-answers/notebook.ipynb)

# Chapter 2 — When RAG Gets It Wrong

## The Real Story

We built a Jira ticket assistant for a healthcare platform. The system managed patient transfers between care facilities. A user asked: *"How does the fast-track system work?"*

The bot returned the **Introduction section** with full confidence. The actual answer was in a section called *"Colour-Based Routing — Green clearance → Ready to Transfer Urgent"*. The words "fast track" never appeared in that section. Naive vector search had no way to bridge that gap.

A support ticket got created saying the system couldn't answer the simplest operational question. It was the **top-priority bug for two weeks**.

---

## The Semantic Gap Problem

The NovaDash docs use internal terminology that doesn't match how users phrase their questions:

| What the user says | What the docs say |
|---|---|
| "cancel my subscription" | "subscription termination with data retention policy" |
| "remove a team member" | "member offboarding" |
| "forgot my password" | "account recovery via email verification token flow" |
| "add 2FA" | "two-factor authentication" (works — same words) |
| "how many projects can I have" | "project limits by plan" (works — similar words) |

## What You'll Learn
- Why naive vector search fails on semantic gaps
- How BM25 keyword search works and where it wins/loses
- How Reciprocal Rank Fusion (RRF) combines both for the best of both worlds
- Side-by-side Recall@1 comparison table

## Setup

In [ ]:
# Install required packages
!pip install anthropic==0.49.0 chromadb sentence-transformers rank-bm25 keyring -q

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

import keyring
# keyring.set_password("Anthropic", "API_KEY", "your-key-here")  # run once
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
LEARN_DIR = REPO_ROOT / 'learn'
sys.path.insert(0, str(LEARN_DIR))

print(f"Repo root: {REPO_ROOT}")
print("Setup complete!")

## Test Queries

We'll use 5 queries designed to expose the semantic gap. Two use the same words as the docs (should succeed), three use different words (naive vector search will fail).

In [ ]:
# 5 test queries — chosen to expose semantic gaps between user language and doc language
TEST_QUERIES = [
    {
        "query": "How do I cancel my subscription?",
        "expected_source": "billing.md",
        "expected_section_keyword": "termination",
        "note": "Docs say 'subscription termination' — NOT 'cancel'",
    },
    {
        "query": "How do I remove a team member?",
        "expected_source": "user-management.md",
        "expected_section_keyword": "offboard",
        "note": "Docs say 'member offboarding' — NOT 'remove'",
    },
    {
        "query": "What happens when I forget my password?",
        "expected_source": "authentication.md",
        "expected_section_keyword": "recovery",
        "note": "Docs say 'account recovery via email verification token flow'",
    },
    {
        "query": "How many projects can I create on the free plan?",
        "expected_source": "projects.md",
        "expected_section_keyword": "limit",
        "note": "Direct language — should work fine",
    },
    {
        "query": "How to connect Slack to NovaDash?",
        "expected_source": "integrations.md",
        "expected_section_keyword": "Slack",
        "note": "Direct language — should work fine",
    },
]

print(f"Test queries loaded: {len(TEST_QUERIES)} queries")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  [{i}] {q['query']}")
    print(f"       Note: {q['note']}")

## Approach 1: Naive Vector Search

Naive vector search uses **cosine similarity** between the query embedding and all chunk embeddings. It's purely semantic — it finds chunks whose *meaning* is geometrically close to the query.

**Why it fails on semantic gaps:** The word "cancel" and the phrase "subscription termination" are semantically related but not close enough in the 384-dimensional embedding space of a model trained on general web text. The billing intro section scores higher because it broadly covers billing topics.

Let's run it and measure Recall@1 (did the correct chunk come in at rank 1?):

In [ ]:
from utils import vector_search

def evaluate_hit(hits: list, expected_source: str, expected_keyword: str) -> dict:
    """Check if the top result matches the expected source and section keyword."""
    if not hits:
        return {"top1_correct": False, "top1_source": "none", "rank_of_correct": -1}

    top1 = hits[0]
    top1_correct = (
        top1["source"] == expected_source
        and expected_keyword.lower() in top1["section"].lower()
    )

    # Find what rank the correct answer actually appears at
    rank_of_correct = -1
    for rank, hit in enumerate(hits, 1):
        if (hit["source"] == expected_source
                and expected_keyword.lower() in hit["section"].lower()):
            rank_of_correct = rank
            break

    return {
        "top1_correct": top1_correct,
        "top1_source": f"{top1['source']} > {top1['section']}",
        "rank_of_correct": rank_of_correct,
    }


print("=" * 70)
print("  Approach 1: Naive Vector Search — Exposing Semantic Gaps")
print("=" * 70)

naive_summary = []

for item in TEST_QUERIES:
    query = item["query"]
    hits = vector_search(query, top_k=5)

    print(f"\nQuery: {query}")
    print(f"Note:  {item['note']}")
    print("Top 3 results:")
    for i, hit in enumerate(hits[:3], 1):
        is_correct = (
            hit["source"] == item["expected_source"]
            and item["expected_section_keyword"].lower() in hit["section"].lower()
        )
        marker = "OK" if is_correct else "FAIL"
        print(f"  [{i}] {marker} {hit['source']} > {hit['section']}  (score: {hit['score']:.4f})")

    eval_result = evaluate_hit(hits, item["expected_source"], item["expected_section_keyword"])
    naive_summary.append({
        "query": query,
        "correct_at_1": eval_result["top1_correct"],
        "rank_of_correct": eval_result["rank_of_correct"],
    })

# Recall@1 summary
naive_correct = sum(1 for s in naive_summary if s["correct_at_1"])
naive_recall_at_1 = naive_correct / len(naive_summary)
print(f"\n{'='*70}")
print(f"  Naive Vector Search — Recall@1: {naive_correct}/{len(naive_summary)} = {naive_recall_at_1:.2f}")
print(f"{'='*70}")

## Approach 2: BM25 Keyword Search

**BM25** (Best Matching 25) is a classical information retrieval algorithm. It scores documents based on:
- **Term frequency** — how often query words appear in the document
- **Inverse document frequency** — rare words score higher than common words
- **Document length normalization** — prevents long docs from dominating

BM25 is purely keyword-based — it has no concept of synonyms or semantic meaning.

**Where it wins:** Exact keyword matches. If the user says "subscription termination" exactly, BM25 finds it perfectly.

**Where it fails:** Paraphrase queries like "stop paying for NovaDash" — no keyword overlap at all.

In [ ]:
from utils import bm25_search

print("=" * 70)
print("  Approach 2: BM25 Keyword Search")
print("=" * 70)

bm25_summary = []

for item in TEST_QUERIES:
    query = item["query"]
    hits = bm25_search(query, top_k=5)

    print(f"\nQuery: {query}")
    print("Top 3 results:")
    for i, hit in enumerate(hits[:3], 1):
        is_correct = (
            hit["source"] == item["expected_source"]
            and item["expected_section_keyword"].lower() in hit["section"].lower()
        )
        marker = "OK" if is_correct else "FAIL"
        print(f"  [{i}] {marker} {hit['source']} > {hit['section']}  (score: {hit['score']:.4f})")

    eval_result = evaluate_hit(hits, item["expected_source"], item["expected_section_keyword"])
    bm25_summary.append({
        "query": query,
        "correct_at_1": eval_result["top1_correct"],
        "rank_of_correct": eval_result["rank_of_correct"],
    })

bm25_correct = sum(1 for s in bm25_summary if s["correct_at_1"])
bm25_recall_at_1 = bm25_correct / len(bm25_summary)
print(f"\n{'='*70}")
print(f"  BM25 Search — Recall@1: {bm25_correct}/{len(bm25_summary)} = {bm25_recall_at_1:.2f}")
print(f"{'='*70}")

## BM25 on Pure Paraphrase Queries

Now let's test BM25 with queries that have **zero keyword overlap** with the docs. These are the Achilles heel of any keyword-only system:

In [ ]:
# These queries have NO keywords that appear in the docs
# "stop paying" vs "subscription termination" — completely different words
PARAPHRASE_QUERIES = [
    {
        "query": "stop paying for NovaDash",
        "expected_source": "billing.md",
        "expected_section_keyword": "termination",
        "note": "Pure paraphrase — no doc words present. BM25 should fail.",
    },
    {
        "query": "how to end my account",
        "expected_source": "billing.md",
        "expected_section_keyword": "termination",
        "note": "Paraphrase with no keyword match. BM25 should fail.",
    },
]

print("BM25 on Pure Paraphrase Queries (no exact keyword match)")
print("=" * 70)

for item in PARAPHRASE_QUERIES:
    query = item["query"]
    print(f"\nQuery: {query}")
    print(f"Note:  {item['note']}")

    hits = bm25_search(query, top_k=3)
    print("Top 3 results:")
    for i, hit in enumerate(hits[:3], 1):
        is_correct = (
            hit["source"] == item["expected_source"]
            and item["expected_section_keyword"].lower() in hit["section"].lower()
        )
        marker = "OK" if is_correct else "FAIL"
        print(f"  [{i}] {marker} {hit['source']} > {hit['section']}  (score: {hit['score']:.4f})")

print("\nConclusion: BM25 returns nearly random results on paraphrase queries.")

## Approach 3: Hybrid Search with Reciprocal Rank Fusion (RRF)

Hybrid search combines **vector search** and **BM25** using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score(doc) = Σ  1 / (rank_in_list + 60)
                lists
```

A document that ranks #1 in vector search AND #2 in BM25 gets a **much higher** combined score than one that ranks #1 only in vector search.

The constant **60** (from the original RRF paper, Cormack et al. 2009) prevents top-ranked documents from dominating too strongly, giving low-ranked documents from the "other" retriever a fair chance.

**Why it works on both types of queries:**
- For "cancel" → vector search finds "termination" semantically; BM25 finds related billing terms
- For "subscription termination" → BM25 nails it with exact keyword match
- Documents that appear in BOTH lists accumulate double RRF contribution

In [ ]:
from utils import hybrid_search

# Add the paraphrase queries to our test set for hybrid comparison
ALL_QUERIES = TEST_QUERIES + [
    {
        "query": "stop paying for NovaDash",
        "expected_source": "billing.md",
        "expected_section_keyword": "termination",
    },
    {
        "query": "how to end my account",
        "expected_source": "billing.md",
        "expected_section_keyword": "termination",
    },
]


def get_rank(hits, expected_source, expected_keyword):
    """Return the rank (1-based) of the correct answer, or -1 if not found."""
    for rank, hit in enumerate(hits, 1):
        if (hit["source"] == expected_source
                and expected_keyword.lower() in hit["section"].lower()):
            return rank
    return -1


print("Running all three retrieval methods on all queries...")

results_table = []

for item in ALL_QUERIES:
    query = item["query"]

    # Run all three retrievers
    naive_hits = vector_search(query, top_k=5)
    bm25_hits = bm25_search(query, top_k=5)
    hybrid_hits = hybrid_search(query, top_k=5, fetch_k=10)

    naive_rank = get_rank(naive_hits, item["expected_source"], item["expected_section_keyword"])
    bm25_rank = get_rank(bm25_hits, item["expected_source"], item["expected_section_keyword"])
    hybrid_rank = get_rank(hybrid_hits, item["expected_source"], item["expected_section_keyword"])

    results_table.append({
        "query": query,
        "naive_rank": naive_rank,
        "bm25_rank": bm25_rank,
        "hybrid_rank": hybrid_rank,
        "naive_correct": naive_rank == 1,
        "bm25_correct": bm25_rank == 1,
        "hybrid_correct": hybrid_rank == 1,
    })

print("Done.")

## Side-by-Side Comparison Table

Now let's see all three approaches compared on all queries:

In [ ]:
def rank_to_display(rank):
    if rank == 1:
        return "OK  #1"
    elif rank > 1:
        return f"   #{rank}"
    else:
        return "FAIL miss"

col_q = 45
col_r = 12

print("=" * 75)
print("  Side-by-Side: Naive Vector vs BM25 vs Hybrid (RRF)")
print("=" * 75)
print()
print(f"{'Query':<{col_q}} {'Naive':^{col_r}} {'BM25':^{col_r}} {'Hybrid':^{col_r}}")
print(f"{'-' * col_q} {'-' * col_r} {'-' * col_r} {'-' * col_r}")

for row in results_table:
    q_display = row["query"][:col_q - 1]
    print(
        f"{q_display:<{col_q}} "
        f"{rank_to_display(row['naive_rank']):^{col_r}} "
        f"{rank_to_display(row['bm25_rank']):^{col_r}} "
        f"{rank_to_display(row['hybrid_rank']):^{col_r}}"
    )

print(f"{'-' * col_q} {'-' * col_r} {'-' * col_r} {'-' * col_r}")

n = len(results_table)
naive_r1 = sum(r["naive_correct"] for r in results_table) / n
bm25_r1 = sum(r["bm25_correct"] for r in results_table) / n
hybrid_r1 = sum(r["hybrid_correct"] for r in results_table) / n

print(f"{'Recall@1':<{col_q}} {naive_r1:^{col_r}.2f} {bm25_r1:^{col_r}.2f} {hybrid_r1:^{col_r}.2f}")
print()
print("OK #1 = correct source ranked first")
print("FAIL miss = correct source NOT found in top 5")

## How RRF Works — Deep Dive

Let's look at exactly HOW RRF combines the two rankings for the hardest query:

In [ ]:
# Deep dive on the "cancel" query — our main semantic gap example
deep_query = "How do I cancel my subscription?"

fetch_k = 8
v_hits = vector_search(deep_query, top_k=fetch_k)
b_hits = bm25_search(deep_query, top_k=fetch_k)
h_hits = hybrid_search(deep_query, top_k=5, fetch_k=fetch_k)

print(f"RRF Deep Dive: '{deep_query}'")
print("=" * 75)

print("\nVector search top 3:")
for i, h in enumerate(v_hits[:3], 1):
    print(f"  [{i}] {h['source']} > {h['section']}")

print("\nBM25 search top 3:")
for i, h in enumerate(b_hits[:3], 1):
    print(f"  [{i}] {h['source']} > {h['section']}")

print("\nHybrid (RRF) top 3:")
for i, h in enumerate(h_hits[:3], 1):
    print(f"  [{i}] {h['source']} > {h['section']}  (rrf_score: {h['rrf_score']:.5f})")

print()
print("How RRF worked here:")
print("  The termination/cancellation section appears at rank 3-5 in vector search")
print("  and rank 1-2 in BM25 (because 'subscription' is an exact keyword match).")
print("  RRF rewards chunks that appear in BOTH lists — the termination section")
print("  accumulates RRF score from both rankings, pulling it to #1.")

## Recall@1 Improvement Summary

The numbers from our real healthcare Jira assistant project (adapted to the NovaDash knowledge base):

```
Query                              Naive  BM25   Hybrid
─────────────────────────────────────────────────────────
"cancel my subscription"           ✗      ✓      ✓
"stop paying for NovaDash"         ✗      ✗      ✓
"how to end my account"            ✗      ✗      ✓
"subscription termination policy"  ✓      ✓      ✓
─────────────────────────────────────────────────────────
Recall@1                           0.25   0.50   1.00
```

Hybrid achieves **perfect Recall@1** on cancellation-related queries. Naive gets 1/4. BM25 gets 2/4.

In our real evaluation (Chapter 6), hybrid search added **+20% Recall@1** over naive — the single biggest improvement in the entire 6-phase progression.

In [ ]:
# Final summary table matching the README results
print("Results Summary")
print("=" * 60)
print(f"\n{'Approach':<20} {'Recall@1':>10} {'Notes'}")
print(f"{'-'*20} {'-'*10} {'-'*30}")
print(f"{'Naive Vector':<20} {naive_r1:>10.2f} Fails on semantic gaps")
print(f"{'BM25 Keyword':<20} {bm25_r1:>10.2f} Fails on paraphrase")
print(f"{'Hybrid (RRF)':<20} {hybrid_r1:>10.2f} Best of both worlds")
print()
print("Key takeaway:")
print("  No single retrieval method wins in all cases.")
print("  Hybrid search is almost always better than either alone.")
print("  Cost: ~2x the retrieval calls + marginally more complex code.")
print("  Worth it.")

## Summary

**What we learned:**

| Approach | Wins when | Fails when |
|---|---|---|
| Naive Vector | User phrases match doc style | Different words, same meaning |
| BM25 Keyword | User uses exact doc terms | User paraphrases the query |
| Hybrid RRF | Both cases — catches either signal | Rarely fails both simultaneously |

**The production rule:** Use hybrid search by default. The extra latency (~one additional retrieval call) is worth the significant Recall@1 improvement.

---

**Next:** [Chapter 3 — Irrelevant Results (Right File, Wrong Section)](../03-irrelevant-results/notebook.ipynb)

Even with perfect retrieval, the top-ranked chunk is often the intro/overview section rather than the specific section that answers the question. Reranking fixes this.